# 01 - Data Collection
## Download de precos historicos das acoes do S&P 500
Periodo: 2010-01-01 ate hoje.
Fonte: Yahoo Finance via yfinance + lista de tickers da Wikipedia.

In [1]:
import pandas as pd
import yfinance as yf
import datetime as dt
import warnings
warnings.filterwarnings("ignore")

### 1.1 Obter lista de tickers do S&P 500
Extraimos a tabela de componentes do S&P 500 direto da Wikipedia.
Alguns tickers usam ponto (ex: BRK.B), mas o Yahoo Finance espera hifen (BRK-B).

A gente precisa da lista de quais empresas fazem parte do S&P 500. O S&P 500 nao e fixo, empresas entram e saem ao longo do tempo. Precisamos dos tickers (codigos de negociacao, tipo AAPL pra Apple, MSFT pra Microsoft) pra saber quais acoes baixar do Yahoo Finance.

Sobre o "por que Wikipedia": nao precisa ser. A Wikipedia so e a fonte mais pratica porque tem uma tabela estruturada e publica com os 503 tickers atuais (503 porque algumas empresas tem duas classes de acoes). Outras opcoes seriam o site oficial da S&P Global (que exige cadastro), APIs pagas como Bloomberg ou Refinitiv, ou ate manter uma lista hardcoded num CSV.

Um ponto importante: a gente esta pegando a composicao **atual** do S&P 500, mas baixando precos desde 2010. Isso gera um vies chamado **survivorship bias**. Empresas que faliram ou foram removidas do indice entre 2010 e hoje nao aparecem na lista atual, entao a gente so treina com as "sobreviventes", o que inflaciona artificialmente os resultados. O notebook original tambem tem esse problema. Corrigir isso de verdade exigiria uma base com o historico de composicao do indice, que e dado pago. Pro escopo de portfolio pessoal, a gente reconhece a limitacao e segue.

In [2]:
import requests
from io import StringIO

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)

sp500_table = pd.read_html(StringIO(response.text))[0]

tickers = sp500_table["Symbol"].str.replace(".", "-", regex=False).tolist()

print(f"Total de tickers: {len(tickers)}")
sp500_table[["Symbol", "Security", "GICS Sector"]].head(10)

Total de tickers: 503


,Symbol,Security,GICS Sector
0,MMM,3M,Industrials
1,AOS,A. O. Smith,Industrials
2,ABT,Abbott Laboratories,Health Care
3,ABBV,AbbVie,Health Care
4,ACN,Accenture,Information Technology
5,ADBE,Adobe Inc.,Information Technology
6,AMD,Advanced Micro Devices,Information Technology
7,AES,AES Corporation,Utilities
8,AFL,Aflac,Financials
9,A,Agilent Technologies,Health Care


### 1.2 Download de precos historicos via yfinance
Baixamos OHLCV (Open, High, Low, Close, Adj Close, Volume) diario de 2010 ate hoje.
Isso pode levar alguns minutos dependendo da conexao.

In [3]:
start_date = "2010-01-01"
end_date = dt.date.today().strftime("%Y-%m-%d")

df = yf.download(
    tickers=tickers,
    start=start_date,
    end=end_date,
    group_by="ticker",
    auto_adjust=False,
    threads=True
)

print(f"Shape: {df.shape}")
print(f"Periodo: {df.index.min()} ate {df.index.max()}")
df.head()

[*********************100%***********************]  503 of 503 completed


Shape: (4102, 3018)
Periodo: 2010-01-04 00:00:00 ate 2026-04-24 00:00:00


Ticker            WFC                                                    VRT  \
Price            Open   High        Low      Close  Adj Close    Volume Open   
Date                                                                           
2010-01-04  27.020000  27.48  26.820000  27.320000  17.937834  39335700  NaN   
2010-01-05  27.270000  28.24  27.240000  28.070000  18.430275  55416000  NaN   
2010-01-06  28.030001  28.33  27.790001  28.110001  18.456539  33237000  NaN   
2010-01-07  28.120001  29.43  27.920000  29.129999  19.126257  61649000  NaN   
2010-01-08  28.900000  29.35  28.600000  28.860001  18.948977  35508700  NaN   

Ticker                     ...        IFF                                \
Price      High Low Close  ...        Low      Close  Adj Close  Volume   
Date                       ...                                            
2010-01-04  NaN NaN   NaN  ...  41.500000  42.009998  29.131941  286000   
2010-01-05  NaN NaN   NaN  ...  41.439999  41.700001  28.916969  348900   
2010-01-06  NaN NaN   NaN  ...  41.509998  41.869999  29.034864  375600   
2010-01-07  NaN NaN   NaN  ...  41.070000  41.549999  28.812952  402000   
2010-01-08  NaN NaN   NaN  ...  40.980000  41.400002  28.708942  249000   

Ticker           RVTY                                                       
Price            Open       High        Low      Close  Adj Close   Volume  
Date                                                                        
2010-01-04  20.760000  20.809999  20.280001  20.639999  18.898886  1880200  
2010-01-05  20.660000  21.059999  20.490000  20.889999  19.127798  1568200  
2010-01-06  20.820000  21.410000  20.799999  21.299999  19.503206  1245300  
2010-01-07  21.799999  22.000000  21.200001  21.240000  19.448265  2581900  
2010-01-08  21.320000  21.510000  21.209999  21.430000  19.622253  2273200  

[5 rows x 3018 columns]

### 1.3 Reestruturar o DataFrame
O yfinance retorna colunas em MultiIndex (ticker, campo).
Vamos empilhar para formato long: indice = (date, ticker), colunas = OHLCV.

Precisamos ates entender e validar o formato atual

In [4]:
print(f"Tipo das colunas: {type(df.columns)}")
print(f"Levels: {df.columns.nlevels}")
print(f"Primeiras 20 colunas: {df.columns.tolist()[:20]}")
print(f"Shape: {df.shape}")

Tipo das colunas: <class 'pandas.MultiIndex'>
Levels: 2
Primeiras 20 colunas: [('WFC', 'Open'), ('WFC', 'High'), ('WFC', 'Low'), ('WFC', 'Close'), ('WFC', 'Adj Close'), ('WFC', 'Volume'), ('VRT', 'Open'), ('VRT', 'High'), ('VRT', 'Low'), ('VRT', 'Close'), ('VRT', 'Adj Close'), ('VRT', 'Volume'), ('PYPL', 'Open'), ('PYPL', 'High'), ('PYPL', 'Low'), ('PYPL', 'Close'), ('PYPL', 'Adj Close'), ('PYPL', 'Volume'), ('ZTS', 'Open'), ('ZTS', 'High')]
Shape: (4102, 3018)


In [6]:
print(df["Price"].unique())
print(df.head(3))

KeyError: 'Price'

In [7]:
df = df.set_index(["Date", "Price"])

df = df.stack().reset_index()
df.columns = ["date", "price_type", "ticker", "value"]

df = df.pivot_table(index=["date", "ticker"], columns="price_type", values="value")
df.columns.name = None
df.columns = [c.lower().replace(" ", "_") for c in df.columns]

df = df[["open", "high", "low", "close", "adj_close", "volume"]]
df = df.dropna(subset=["close"])
df = df.sort_index()

print(f"Shape final: {df.shape}")
print(f"Tickers unicos: {df.index.get_level_values('ticker').nunique()}")
df.head(10)

KeyError: "None of ['Date', 'Price'] are in the columns"

### 1.4 Verificacao basica de qualidade

In [8]:
print("Valores nulos por coluna:")
print(df.isnull().sum())
print(f"\nLinhas duplicadas: {df.index.duplicated().sum()}")

records_per_ticker = df.groupby("ticker").size()
print(f"\nRegistros por ticker (min): {records_per_ticker.min()}")
print(f"Registros por ticker (max): {records_per_ticker.max()}")
print(f"Registros por ticker (mediana): {records_per_ticker.median():.0f}")

Valores nulos por coluna:
Ticker  Price    
WFC     Open         0
        High         0
        Low          0
        Close        0
        Adj Close    0
                    ..
RVTY    High         0
        Low          0
        Close        0
        Adj Close    0
        Volume       0
Length: 3018, dtype: int64

Linhas duplicadas: 0


KeyError: 'ticker'

Dados limpos. Zero nulos, zero duplicatas. A maioria dos tickers tem o historico completo (4102 dias desde 2010). Os que tem menos (minimo 124) sao empresas que entraram no S&P 500 recentemente, tipo Carvana ou AppLovin. Normal.

### 1.5 Salvar dados brutos

In [ ]:
df.to_parquet("../data/raw/sp500_daily_prices.parquet")
print("Salvo em data/raw/sp500_daily_prices.parquet")
print(f"Tamanho: {df.shape[0]:,} linhas x {df.shape[1]} colunas")

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.